In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Exécuter des circuits dynamiques

import Tabs from '@theme/Tabs';
import TabItem from '@theme/TabItem';



<Accordion>
<AccordionItem title="Versions des packages">

Le code sur cette page a été développé en utilisant les exigences suivantes.
Nous recommandons d'utiliser ces versions ou des versions plus récentes.

```
qiskit[all]~=2.4.0
qiskit-ibm-runtime~=0.46.1
```
</AccordionItem>
</Accordion>
Les circuits dynamiques sont des outils puissants avec lesquels tu peux mesurer des qubits au milieu d'une exécution de circuit quantique et ensuite effectuer des opérations de logique classique dans le circuit, sur la base du résultat de ces mesures au milieu du circuit. Ce processus est également connu sous le nom de _retour d'information classique_. Bien que nous soyons aux débuts de la compréhension de la meilleure façon de profiter des circuits dynamiques, la communauté de recherche quantique a déjà identifié un certain nombre de cas d'utilisation, tels que les suivants :

* Préparation efficace des états quantiques, comme l'[état GHZ](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339), l'[état W](https://arxiv.org/abs/2403.07604) (pour plus d'informations sur l'état W, se référer également à ["State preparation by shallow circuits using feed forward"](https://arxiv.org/abs/2307.14840)), et une large classe d'[états produits matriciels](https://arxiv.org/abs/2404.16083)
* [Intrication longue portée efficace](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) entre qubits sur la même puce en utilisant des circuits peu profonds
* [Échantillonnage efficace de circuits de type IQP](https://arxiv.org/pdf/2505.04705)

Ces améliorations apportées par les circuits dynamiques, cependant, s'accompagnent de compromis. Les mesures au milieu du circuit et les opérations classiques ont généralement un temps d'exécution plus long que les gates à deux qubits, et cette augmentation de temps pourrait annuler les avantages d'une profondeur de circuit réduite. Par conséquent, la réduction de la durée des mesures au milieu du circuit est un axe d'amélioration prioritaire alors qu'IBM Quantum&reg; publie la [nouvelle version](/guides/latest-updates#new-version-dynamic-circuits) des circuits dynamiques. Pour d'autres restrictions lors de l'utilisation de circuits dynamiques, consulte le tableau de compatibilité des fonctionnalités [Estimator](/guides/estimator-options#options-compatibility-table) ou [Sampler](/guides/sampler-options#options-compatibility-table).

La [spécification OpenQASM 3](https://openqasm.com/language/classical.html#looping-and-branching) définit un certain nombre de structures de flux de contrôle, mais Qiskit Runtime ne prend actuellement en charge que l'instruction conditionnelle `if`. Dans le SDK Qiskit, cela correspond à la méthode [if_test](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#if_test) sur [QuantumCircuit](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit). Cette méthode renvoie un [gestionnaire de contexte](https://docs.python.org/3/reference/datamodel.html#with-statement-context-managers) et est généralement utilisée dans une instruction `with`. Ce guide décrit comment utiliser cette instruction conditionnelle.

> **Note:** Les exemples de code dans ce guide utilisent l'instruction de mesure standard pour les mesures au milieu du circuit. Cependant, il est recommandé d'utiliser l'instruction `MidCircuitMeasure` à la place, si le Backend le prend en charge. Consulte la [section Mesures au milieu du circuit](#midcircuit) pour plus de détails.
## Trouver les Backends qui prennent en charge les circuits dynamiques
Pour trouver tous les Backends auxquels ton compte peut accéder et qui prennent en charge les circuits dynamiques, exécute du code comme le suivant. Cet exemple suppose que tu as [sauvegardé tes identifiants de connexion](/guides/save-credentials). Tu pourrais également [spécifier explicitement les identifiants](/guides/initialize-account#explicit) lors de l'initialisation de ton compte de service Qiskit Runtime. Cela te permettrait de voir les Backends disponibles sur une instance ou un type de plan spécifique, par exemple.

> **Note:** - Les Backends disponibles pour le compte dépendent de l'instance spécifiée dans les identifiants.
> - La nouvelle version des circuits dynamiques est maintenant disponible pour tous les utilisateurs sur tous les Backends. Consulte [l'annonce](/guides/latest-updates#new-dynamic-circuits) pour plus de détails.

In [ ]:
# This cell is hidden from users. It hides all those "...instance was not set..." warnings.
import warnings

warnings.filterwarnings("ignore", message=".*Instance was not set*")

In [2]:
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
dc_backends = service.backends(dynamic_circuits=True)
print(dc_backends)

[<IBMBackend('ibm_pittsburgh')>, <IBMBackend('ibm_kingston')>, <IBMBackend('ibm_marrakesh')>, <IBMBackend('ibm_fez')>, <IBMBackend('ibm_boston')>]


<span id="midcircuit"></span>
## Mid-circuit measurements

Prior to `qiskit-ibm-runtime` v0.43.0, `measure` was the only measurement instruction in Qiskit. Mid-circuit measurements, however, have different tuning requirements than _terminal_ measurements (measurements that happen at the end of a circuit). For example, you need to consider the instruction duration when tuning a mid-circuit measurement because longer instructions cause noisier circuits. You don't need to consider instruction duration for terminal measurements because there are no instructions after terminal measurements.

<Admonition type="note">
The `MidCircuitMeasure` instruction maps to the `measure_2` instruction reported in the backend's `supported_instructions`. However,  `measure_2` is not supported on all backends. Use `service.backends(filters=lambda b: "measure_2" in b.supported_instructions)` to find backends that support it.  New measurements might be added in the future, but this is not guarenteed.
</Admonition>

### `MidCircuitMeasure` method

In `qiskit-ibm-runtime` v0.43.0, the `MidCircuitMeasure` instruction was introduced. As the name suggests, it is a new measurement instruction that is optimized for mid-circuit on IBM&reg; QPUs.  While you can use `QuantumCircuit.measure` for a mid-circuit measurement, because of its design, `MidCircuitMeasure` is typically a better choice.  For example, it adds less overhead to your circuit than when using `QuantumCircuit.measure`.

In [3]:
from qiskit import QuantumCircuit
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.circuit import MidCircuitMeasure

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, dynamic_circuits=True
)

circ = QuantumCircuit(2, 2)
circ.x(0)
circ.append(MidCircuitMeasure(), [0], [0])
# circ.measure([0], [0])
# circ.measure_all()
print(circ.draw(cregbundle=False))

     ┌───┐┌────────────┐
q_0: ┤ X ├┤0           ├
     └───┘│            │
q_1: ─────┤  Measure_2 ├
          │            │
c_0: ═════╡0           ╞
          └────────────┘
c_1: ═══════════════════
                        


<Admonition type="attention" title="Important notes">

* There must be at least one classical register in order to use measurements.
* The Sampler primitive requires circuit measurements. You can add circuit measurements with the Estimator primitive, but they are ignored.

</Admonition>

## `Store`

With `qiskit-ibm-runtime` version 0.47.0 or later, you can use the [`store`](/docs/api/qiskit/circuit#store) instruction to save the result of a classical expression, if that expression will be used repeatedly. Operations are automatically parallelized, making your code significantly more efficient at runtime.

For more information, see the [Classical feedforward and control flow](/docs/guides/classical-feedforward-and-control-flow#store) guide.

<Admonition type="note">
    When you use `store` to save a value to a classical register on a real backend, that value is only saved in memory during execution and not copied or returned in the job result.

    For example, in the following code, `temp` has the same value as `creg` during execution, and the `if_test` works as expected. But after the job finishes, the `temp` BitArray returned in the job result **does not** contain the value of `creg`.  That is, `job.result()[0].data.temp` is `0`.

    ```python
    creg = ClassicalRegister(3, "c")
    temp = ClassicalRegister(3, "temp")
    ...
    qc.store(temp, creg)
    with circuit.if_test((temp, 0b001)):
        ...
    ```
</Admonition>

## Full example

The following code creates and runs a dynamic circuit on IBM&reg; hardware.

In [4]:
from qiskit_ibm_runtime import SamplerV2, QiskitRuntimeService
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.transpiler import generate_preset_pass_manager

service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True, simulator=False, dynamic_circuits=True
)

# Create a dynamic circuit

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
qc = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

qc.h(q0)
qc.measure(q0, c0)
with qc.if_test((c0, 1)):
    qc.x(q0)
qc.measure(q0, c0)


# Convert to an ISA circuit for the given backend

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

# Generate samplers for backend targets
sampler = SamplerV2(backend)

# Submit jobs
sampler_job = sampler.run([isa_circuit])
result = sampler_job.result()

print(
    f">>> {' Job ID:':<10}  {sampler_job.job_id()} ({sampler_job.status()})"
)

>>>  Job ID:    d88cakp789is7391vq0g (DONE)


<span id="midcircuit"></span>
## Mesures au milieu du circuit
Avant `qiskit-ibm-runtime` v0.43.0, `measure` était la seule instruction de mesure dans Qiskit. Les mesures au milieu du circuit, cependant, ont des exigences de réglage différentes des mesures _terminales_ (mesures qui se produisent à la fin d'un circuit). Par exemple, tu dois prendre en compte la durée de l'instruction lors du réglage d'une mesure au milieu du circuit car des instructions plus longues provoquent des circuits plus bruyants. Tu n'as pas besoin de prendre en compte la durée de l'instruction pour les mesures terminales car il n'y a pas d'instructions après les mesures terminales.

> **Note:** L'instruction `MidCircuitMeasure` correspond à l'instruction `measure_2` signalée dans les `supported_instructions` du Backend. Cependant, `measure_2` n'est pas pris en charge sur tous les Backends. Utilise `service.backends(filters=lambda b: "measure_2" in b.supported_instructions)` pour trouver les Backends qui le prennent en charge. De nouvelles mesures pourraient être ajoutées à l'avenir, mais ce n'est pas garanti.

### Méthode `MidCircuitMeasure`
Dans `qiskit-ibm-runtime` v0.43.0, l'instruction `MidCircuitMeasure` a été introduite. Comme son nom l'indique, c'est une nouvelle instruction de mesure optimisée pour les mesures au milieu du circuit sur les QPU IBM&reg;. Bien que tu puisses utiliser `QuantumCircuit.measure` pour une mesure au milieu du circuit, en raison de sa conception, `MidCircuitMeasure` est généralement un meilleur choix. Par exemple, il ajoute moins de surcharge à ton circuit que lors de l'utilisation de `QuantumCircuit.measure`.